In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ["TF_CUDNN_USE_AUTOTUNE"] = "0"
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
tf.config.optimizer.set_jit(True)  # Enable XLA

E0000 00:00:1746015410.188478   54528 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746015410.193658   54528 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
import numpy as np
from glob import glob
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, initializers, regularizers
from keras.callbacks import LearningRateScheduler, EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.mixed_precision import set_global_policy
policy = tf.keras.mixed_precision.Policy('mixed_float16')
tf.keras.mixed_precision.set_global_policy(policy)
print("Mixed precision enabled with policy:", policy.name)

Mixed precision enabled with policy: mixed_float16


In [33]:
T_FIXED = 432
mean_global = -38.622053581065884
std_global = 13.92796650578306

### Helper functions

In [6]:
def get_feature_description():
    feature_description = {
        'mel': tf.io.FixedLenFeature([], tf.string),
        'labels': tf.io.VarLenFeature(tf.int64),
        'song': tf.io.FixedLenFeature([], tf.string),
        'segment_idx': tf.io.FixedLenFeature([], tf.int64),
        'total_segments': tf.io.FixedLenFeature([], tf.int64),
    }
    return feature_description

In [7]:
def get_labels(file):
    dataset = tf.data.TFRecordDataset(file)
    feature_description = {
        'labels': tf.io.VarLenFeature(tf.int64)
    }
    example = next(iter(dataset))

    parsed = tf.io.parse_single_example(example, feature_description)
    labels = tf.sparse.to_dense(parsed['labels']).numpy()
    return labels

In [8]:
def get_song_id(file):
    ds = tf.data.TFRecordDataset(file)
    feat = {'song': tf.io.FixedLenFeature([], tf.string)}
    raw = next(iter(ds))
    parsed = tf.io.parse_single_example(raw, feat)
    return parsed['song'].numpy().decode('utf-8')

In [9]:
def parse_tfrecord_fn(example):
    feature_description = get_feature_description()
    example = tf.io.parse_single_example(example, feature_description)
    
    # Directly parse the float32 tensor
    mel_spec = tf.io.parse_tensor(example['mel'], out_type=tf.float32)
    
    current_shape = tf.shape(mel_spec)
    
    if tf.rank(mel_spec) != 2:
        # If not a 2D tensor, reshape it to [96, ?]
        mel_spec = tf.reshape(mel_spec, [96, -1])
        current_shape = tf.shape(mel_spec)
    
    # Crop or pad the time dimension to 650
    if current_shape[1] > T_FIXED:
        # Crop to [96, 432]
        mel_spec = mel_spec[:, :T_FIXED]
    else:
        # Pad to [96, 432]
        paddings = [[0, 0], [0, T_FIXED - current_shape[1]]]
        mel_spec = tf.pad(mel_spec, paddings, "CONSTANT", constant_values=0)
    
    mel_spec = tf.ensure_shape(mel_spec, [96, T_FIXED])

    # normalization
    mel_spec = (mel_spec - mean_global) / (std_global + 1e-6)
    
    # Add channel dimension for CNN input
    mel_spec = tf.expand_dims(mel_spec, axis=-1)
    
    labels = tf.sparse.to_dense(example['labels'])
    labels = tf.ensure_shape(labels, [8])

    
    return mel_spec, labels

In [10]:
def prepare_dataset(tfrecord_pattern, batch_size=32, is_training=True, aug = 'light'):
    
    dataset = tf.data.Dataset.list_files(tfrecord_pattern, shuffle=is_training)

    dataset = dataset.interleave(
        tf.data.TFRecordDataset,
        cycle_length=6,
        num_parallel_calls=6,
        deterministic= False
    )

    dataset = dataset.map(parse_tfrecord_fn, num_parallel_calls=8)
    #dataset = dataset.cache()
    
    if is_training:
        if aug == 'light':
            aug_fn = make_augment(20,  8,  0.1,  0.005,  20)
        elif aug == 'normal':
            aug_fn = make_augment(40, 16, 0.3, 0.02, 40)
        else:  #strong
            aug_fn = make_augment(60, 16, 0.4, 0.1, 60)

        dataset = dataset.map(aug_fn, num_parallel_calls=6)
        dataset = dataset.shuffle(50000, reshuffle_each_iteration=True)
        
    dataset = dataset.batch(batch_size, drop_remainder=is_training)

    dataset = dataset.map(
        lambda x, y: (x, {'aux_out': y, 'binary_output': y}),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    
    if is_training:
        dataset = dataset.repeat()
    
    dataset = dataset.prefetch(1)
    
    return dataset

### Augmentations

In [11]:
def make_augment(max_frames, max_bins, gain, noise_std, crop):
    def augment(m, labels):
        orig_e = tf.reduce_mean(tf.square(m))
        T = tf.shape(m)[1]

        # — Time Mask —
        if max_frames > 0:
            def do_time_mask():
                mm = time_mask(m, max_frames)
                e = tf.reduce_mean(tf.square(mm))
                return mm * tf.sqrt(orig_e/(e + 1e-8))
            m = tf.cond(
                tf.random.uniform([]) < 0.8,
                true_fn=do_time_mask,
                false_fn=lambda: m
            )

        # — Freq Mask —
        if max_bins > 0:
            def do_freq_mask():
                mm = freq_mask(m, max_bins)
                e = tf.reduce_mean(tf.square(mm))
                return mm * tf.sqrt(orig_e/(e + 1e-8))
            m = tf.cond(
                tf.random.uniform([]) < 0.8,
                true_fn=do_freq_mask,
                false_fn=lambda: m
            )

        # — Random Gain —
        if gain > 0:
            def do_gain():
                return random_gain(m, 1.0 - gain, 1.0 + gain)
            m = tf.cond(
                tf.random.uniform([]) < 0.7,
                true_fn=do_gain,
                false_fn=lambda: m
            )

        # — Time Shift —
        m = tf.cond(
            tf.random.uniform([]) < 0.6,
            true_fn=lambda: random_time_shift(m, max_shift=10),
            false_fn=lambda: m
        )

        # — Noise + Renorm —
        if noise_std > 0:
            def do_noise():
                std = tf.random.uniform([], noise_std/2, noise_std)
                mm = m + tf.random.normal(tf.shape(m), 0., std, dtype=m.dtype)
                e = tf.reduce_mean(tf.square(mm))
                return mm * tf.sqrt(orig_e/(e + 1e-8))
            m = tf.cond(
                tf.random.uniform([]) < 1.0,
                true_fn=do_noise,
                false_fn=lambda: m
            )

        # — Time Crop + Renorm + Pad —
        if crop > 0:
            def do_crop():
                win = tf.random.uniform([], T - crop, T, dtype=tf.int32)
                start = tf.random.uniform([], 0, T - win, dtype=tf.int32)
                cm = m[:, start:start + win, :]
                e = tf.reduce_mean(tf.square(cm))
                cm = cm * tf.sqrt(orig_e/(e + 1e-8))
                return tf.pad(cm, [[0,0], [0, T - win], [0,0]])
            m = tf.cond(
                tf.random.uniform([]) < 0.45,
                true_fn=do_crop,
                false_fn=lambda: m
            )

        m = tf.ensure_shape(m, [96, T_FIXED, 1])
        return m, labels

    return augment

In [12]:
def time_mask(x, max_frames=40):
    """zero-out a random consecutive time stripe"""
    T  = tf.shape(x)[1]
    t  = tf.random.uniform([], 0, max_frames, tf.int32)
    t0 = tf.random.uniform([], 0, T - t,      tf.int32)
    zeros = tf.zeros_like(x[:, t0:t0+t, :])
    return tf.concat([x[:, :t0, :], zeros, x[:, t0+t:, :]], axis=1)

In [13]:
def freq_mask(x, max_bins=16):
    """zero-out a random consecutive mel-bin stripe"""
    F  = tf.shape(x)[0]
    f  = tf.random.uniform([], 0, max_bins, tf.int32)
    f0 = tf.random.uniform([], 0, F - f,      tf.int32)
    zeros = tf.zeros_like(x[f0:f0+f, :, :])
    return tf.concat([x[:f0, :, :], zeros, x[f0+f:, :, :]], axis=0)

In [14]:
def random_gain(x, min_gain=0.9, max_gain=1.1):
    '''small volume-like tweaks'''
    gain = tf.random.uniform([], min_gain, max_gain)
    return x * gain

In [15]:
def random_time_shift(x, max_shift=10):
    '''roll time axis and wrap it around'''
    shift = tf.random.uniform([], -max_shift, max_shift, dtype=tf.int32)
    return tf.roll(x, shift=shift, axis=1)

In [16]:
from sklearn.model_selection import train_test_split
from collections import Counter

In [17]:
batch_size = 10

In [18]:
files_dir = tf.io.gfile.glob(
    "/home/georgios/Music Analysis/dataset_creation/tfrecord_dataset_10s/train/*.tfrecord"
)
labels = np.array([get_labels(f) for f in files_dir])
all_files = np.array(files_dir)

I0000 00:00:1746017605.160547   54528 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5520 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [19]:
song_ids = [get_song_id(f) for f in all_files]

In [20]:
# map song_id to indices
song_to_idxs = {}
for i, sid in enumerate(song_ids):
    song_to_idxs.setdefault(sid, []).append(i)

In [21]:
# compute initial val_indices by label coverage
n_total = len(all_files)
val_split = 0.10
n_each = int(n_total * (val_split/2))

In [22]:
start = np.arange(n_each)
end   = np.arange(n_total-n_each, n_total)
val_set = set(np.concatenate([start, end]))

In [23]:
for g in range(labels.shape[1]):
    vl = labels[list(val_set)]
    if vl[:,g].sum() == 0:
        for i in range(n_total):
            if i not in val_set and labels[i, g] == 1:
                val_set.add(i)
                break

In [24]:
# expand to include all segments of each chosen song
expanded = set()
for i in val_set:
    sid = song_ids[i]
    expanded.update(song_to_idxs[sid])

In [25]:
val_indices = np.array(sorted(expanded))
train_indices = np.array([i for i in range(n_total) if i not in val_indices])

In [26]:
# final splits
train_files = all_files[train_indices]
val_files   = all_files[val_indices]

print("Train samples:", len(train_files))
print("Val   samples:", len(val_files))

Train samples: 63
Val   samples: 8


In [34]:
inspect_dataset = prepare_dataset(train_files, batch_size= batch_size, aug='light')#start with light augmentation

In [35]:
import librosa
import librosa.display
from IPython.display import Audio

sr = 22050
n_fft = 2048
hop_length = 512
n_mels = 96

In [36]:
def convert_mel_to_audio(mel_spectrogram):
    if not isinstance(mel_spectrogram, np.ndarray):
        mel_spectrogram = mel_spectrogram.numpy()
    mel_to_linear = librosa.feature.inverse.mel_to_stft(
        mel_spectrogram,
        sr=sr,
        n_fft=n_fft
    )
    
    # Reconstruct audio signal using Griffin-Lim algorithm
    audio = librosa.griffinlim(
        mel_to_linear,
        n_iter=32,
        hop_length=hop_length,
        win_length=n_fft
    )
    
    return audio
    Audio(audio, rate=sr)

In [37]:
def inspect_augmented_versions(X, y, target_genre):
    # Find genre index
    try:
        genre_idx = genre_names.index(target_genre)
    except ValueError:
        raise ValueError(f"Genre '{target_genre}' not found. Available genres: {genre_names}")

    # Find samples matching the genre
    matching_indices = [i for i in range(len(y)) if y[i][genre_idx] == 1]

    if not matching_indices:
        print(f"No samples found for genre '{target_genre}' in this batch.")
        return

    # Pick first matching sample
    sample_index = matching_indices[0]

    base_song = X[sample_index]
    base_label = y[sample_index]

    mel_spectrogram = base_song.numpy().squeeze(-1)  # (96, time)

    base_genres = [genre_names[i] for i in range(len(genre_names)) if base_label[i] == 1]

    print(f"Sample Genres: {', '.join(base_genres)}")

    # Define augmentations
    augmentations = [
        ('Original', {}),
        ('Time Mask', {'time_mask_frames': 100}),
        ('Freq Mask', {'freq_mask_bins': 60}),
        ('Random Gain', {'gain_range': 0.5}),
        ('Normalize Noise', {'normalize_std': 0.5}),
        ('Time Crop', {'time_crop_max': 200}),
    ]

    fig, axs = plt.subplots(1, len(augmentations), figsize=(20, 3))

    for idx, (title, aug_params) in enumerate(augmentations):
        m = tf.convert_to_tensor(mel_spectrogram[..., np.newaxis])  # (96, time, 1)
        m, _ = light_augment(m, labels=None, **aug_params)

        m = m.numpy().squeeze(-1)

        audio = convert_mel_to_audio(m)

        axs[idx].set_title(title)
        axs[idx].axis('off')

        display(Audio(audio, rate=sr))

    plt.show()

In [38]:
genre_names = ['LAIKO', 'REMPETIKO', 'ENTEXNO', 'ROCK', 'Mod LAIKO', 'POP', 'ENALLAKTIKO', 'HIPHOP/RNB']

In [ ]:
for X, y in inspect_dataset.take(1):
    base_song = X[0]
    mel_spectrogram = base_song.numpy().squeeze(-1)  # (96, time)
    audio = convert_mel_to_audio(mel_spectrogram)
    display(Audio(audio, rate=sr))

In [ ]:
for X_batch, y_batch in val_dataset.take(1):
    inspect_augmented_versions(X_batch, y_batch['binary_output'], target_genre='ROCK')

In [ ]:
for X, y in train_dataset.take(1):
    print(X.dtype)